In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import pickle
import warnings
import os
from typing import Dict, List, Tuple, Optional

warnings.filterwarnings('ignore')

from kmrf import KMRF
from KMRF_training_config import *
from SIMULATOR import SIMULATOR
from OPTIMIZER_INPUTS import OPTIMIZER_INPUTS
from BACKTEST import BACKTEST, generate_rebalance_dates


# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  Pandas version: 2.3.3
  NumPy version: 2.3.4


In [3]:
preds_df = pd.read_pickle('data/multi_horizon_predictions/SPDR_S&P_500_ETF_multi_horizon_predictions_original.pkl')

In [11]:
preds_df.xs('2025-03-07', level=0, drop_level=False)

model_end_date  P(LV_Bull)  P(LV_Bear)  P(HV_Bull)  \
date       horizon                                                      
2025-03-07 1             20250205    0.957036    0.010690    0.024074   
           2             20250205    0.917675    0.022068    0.039833   
           3             20250205    0.900857    0.034594    0.036421   
           4             20250205    0.877227    0.048908    0.034501   
           5             20250205    0.876484    0.052248    0.026561   
           6             20250205    0.778547    0.107098    0.039551   
           7             20250205    0.836449    0.088068    0.024721   
           8             20250205    0.759371    0.135166    0.029902   
           9             20250205    0.676045    0.188969    0.034238   
           10            20250205    0.622218    0.239324    0.036564   
           11            20250205    0.549006    0.263345    0.040616   
           12            20250205    0.535852    0.322456    0.036449   
           13            20250205    0.541755    0.276551    0.034751   
           14            20250205    0.558949    0.257659    0.037252   
           15            20250205    0.570134    0.278867    0.025796   
           16            20250205    0.522343    0.271580    0.033865   
           17            20250205    0.492817    0.326226    0.029832   
           18            20250205    0.554082    0.297506    0.026522   
           19            20250205    0.615778    0.233489    0.025059   
           20            20250205    0.682067    0.175783    0.019927   
           21            20250205    0.722452    0.159507    0.018439   

                    P(HV_Bear)  
date       horizon              
2025-03-07 1          0.008200  
           2          0.020424  
           3          0.028128  
           4          0.039364  
           5          0.044706  
           6          0.074805  
           7          0.050762  
           8          0.075562  
           9          0.100748  
           10         0.101895  
           11         0.147032  
           12         0.105244  
           13         0.146944  
           14         0.146139  
           15         0.125203  
           16         0.172212  
           17         0.151125  
           18         0.121890  
           19         0.125674  
           20         0.122222  
           21         0.099602

In [21]:
preds_df.loc['2025-03-10':].index.get_level_values(0).unique()

DatetimeIndex(['2025-03-10', '2025-03-11', '2025-03-12', '2025-03-13',
               '2025-03-14', '2025-03-17', '2025-03-18', '2025-03-19',
               '2025-03-20', '2025-03-21',
               ...
               '2025-10-20', '2025-10-21', '2025-10-22', '2025-10-23',
               '2025-10-24', '2025-10-27', '2025-10-28', '2025-10-29',
               '2025-10-30', '2025-10-31'],
              dtype='datetime64[ns]', name='date', length=165, freq=None)

In [13]:
preds_df.xs('2025-03-10', level=0, drop_level=False)

model_end_date  P(LV_Bull)  P(LV_Bear)  P(HV_Bull)  \
date       horizon                                                      
2025-03-10 1             20250307    0.526910    0.415306    0.039317   
           2             20250307    0.224057    0.724967    0.028541   
           3             20250307    0.192725    0.748264    0.029216   
           4             20250307    0.176053    0.754658    0.024750   
           5             20250307    0.196387    0.728850    0.022226   
           6             20250307    0.245138    0.651621    0.026962   
           7             20250307    0.387406    0.538284    0.019710   
           8             20250307    0.275435    0.645165    0.018513   
           9             20250307    0.346309    0.549280    0.021065   
           10            20250307    0.270644    0.640329    0.019067   
           11            20250307    0.326699    0.587346    0.017922   
           12            20250307    0.298584    0.601596    0.018915   
           13            20250307    0.305979    0.590443    0.019767   
           14            20250307    0.292366    0.615208    0.015672   
           15            20250307    0.371145    0.535749    0.016748   
           16            20250307    0.387225    0.540047    0.012912   
           17            20250307    0.470246    0.443695    0.013847   
           18            20250307    0.454094    0.484648    0.011010   
           19            20250307    0.471374    0.441422    0.012756   
           20            20250307    0.366980    0.532853    0.013728   
           21            20250307    0.416777    0.503342    0.010489   

                    P(HV_Bear)  
date       horizon              
2025-03-10 1          0.018467  
           2          0.022436  
           3          0.029795  
           4          0.044539  
           5          0.052537  
           6          0.076280  
           7          0.054600  
           8          0.060887  
           9          0.083346  
           10         0.069960  
           11         0.068033  
           12         0.080904  
           13         0.083811  
           14         0.076754  
           15         0.076358  
           16         0.059815  
           17         0.072212  
           18         0.050247  
           19         0.074448  
           20         0.086439  
           21         0.069391

In [24]:
preds_df.xs('2025-03-14', level=0, drop_level=False)

model_end_date  P(LV_Bull)  P(LV_Bear)  P(HV_Bull)  \
date       horizon                                                      
2025-03-14 1             20250307    0.036345    0.920582    0.021061   
           2             20250307    0.030667    0.913507    0.018265   
           3             20250307    0.041090    0.875119    0.019269   
           4             20250307    0.032732    0.881976    0.012692   
           5             20250307    0.037648    0.847266    0.012204   
           6             20250307    0.041585    0.820639    0.011939   
           7             20250307    0.051431    0.852403    0.009363   
           8             20250307    0.053042    0.867070    0.009267   
           9             20250307    0.048155    0.874394    0.008561   
           10            20250307    0.037522    0.880684    0.007637   
           11            20250307    0.059128    0.856665    0.008970   
           12            20250307    0.051367    0.861744    0.009555   
           13            20250307    0.058303    0.844535    0.012312   
           14            20250307    0.065262    0.842442    0.009845   
           15            20250307    0.039162    0.902517    0.007559   
           16            20250307    0.055455    0.879323    0.008093   
           17            20250307    0.107850    0.798820    0.010733   
           18            20250307    0.092654    0.837473    0.008716   
           19            20250307    0.086182    0.842123    0.006908   
           20            20250307    0.099819    0.804371    0.007865   
           21            20250307    0.106720    0.823826    0.006900   

                    P(HV_Bear)  
date       horizon              
2025-03-14 1          0.022013  
           2          0.037561  
           3          0.064523  
           4          0.072599  
           5          0.102882  
           6          0.125837  
           7          0.086803  
           8          0.070621  
           9          0.068890  
           10         0.074156  
           11         0.075237  
           12         0.077334  
           13         0.084849  
           14         0.082450  
           15         0.050762  
           16         0.057130  
           17         0.082596  
           18         0.061157  
           19         0.064787  
           20         0.087945  
           21         0.062554